# VoiceForensics — GPU training (Colab / Kaggle / RunPod)

Turnkey notebook to train the **weight-gated** neural backends (RawNet2 / AASIST) on a
labelled anti-spoofing dataset, fit calibration, and export checkpoints the engine can
load directly.

> **Defensive security tooling.** This trains a *detector* of synthetic speech. Engine
> outputs become `baseline_only=False` only after you load the trained weights from here.

**Steps:** install → check GPU → prepare data (`real/` + `fake/`) → train → export weights.

Pick a **GPU** runtime (Colab: Runtime ▸ Change runtime type ▸ GPU).

## 1. Install (CUDA PyTorch + VoiceForensics)

In [ ]:
# ffmpeg is required for decoding/metadata.
!apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null

# CUDA build of PyTorch (cu121). On CPU-only runtimes use the cpu index instead.
!pip -q install torch --index-url https://download.pytorch.org/whl/cu121

# Install VoiceForensics. Replace with your repo URL/branch, or `pip install -e .`
# if you cloned it next to this notebook.
REPO = "https://github.com/alanwasdead/VoiceForensics.git"
BRANCH = "cursor/core-detection-engine-6766"
!pip -q install "git+{REPO}@{BRANCH}"  # noqa


## 2. Verify the GPU

In [ ]:
import torch

from voiceforensics.training.train import resolve_device

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("resolved device:", resolve_device("auto"))

## 3. Prepare the dataset

Arrange audio into `real/` and `fake/` subdirectories. For ASVspoof, convert the
protocol labels to this layout, e.g.:

```python
from voiceforensics.training.datasets import asvspoof_protocol
labels = asvspoof_protocol('/data/ASVspoof2019_LA_cm_protocols/...trn.txt')
# then symlink/copy each utterance into real/ (bonafide) or fake/ (spoof)
```

Below we build a tiny **synthetic** dataset just so the notebook runs end-to-end; replace
`DATASET` with your real data directory for an actual model.

In [ ]:
# Demo-only synthetic dataset (NOT real speech). Replace DATASET with real data.
from pathlib import Path

import numpy as np
import soundfile as sf

DATASET = Path('demo_dataset')
from voiceforensics.features.spectral import log_mel_spectrogram  # noqa: F401 (warms imports)


def _tone(freq, n=48000, sr=16000, jitter=0.0, lowpass=None, seed=0):
    rng = np.random.default_rng(seed); t = np.arange(n)/sr
    f0 = freq + jitter*np.cumsum(rng.normal(0,1,n))/sr
    sig = np.sin(2*np.pi*np.cumsum(f0)/sr)
    for k in range(2, 25): sig += (1.0/k)*np.sin(2*np.pi*k*np.cumsum(f0)/sr)
    env = (0.5+0.5*np.sin(2*np.pi*3*t))**2; sig = sig*env + rng.normal(0,0.001,n)
    if lowpass:
        from scipy.signal import butter, sosfiltfilt
        sig = sosfiltfilt(butter(4, lowpass/(sr/2), output='sos'), sig)
    return (0.9*sig/np.max(np.abs(sig))).astype('float32')

for sub in ('real','fake'): (DATASET/sub).mkdir(parents=True, exist_ok=True)
for i in range(8):
    sf.write(DATASET/'real'/f'r{i}.wav', _tone(120, jitter=0.6, seed=i), 16000)
    sf.write(DATASET/'fake'/f'f{i}.wav', _tone(130, jitter=0.0, lowpass=7000, seed=99+i), 16000)
print('demo dataset ready at', DATASET.resolve())

## 4. Train both backends + calibrate

In [ ]:
# For a real run: increase --epochs, point at your dataset, use a GPU runtime.
!python -m voiceforensics.__main__ --help >/dev/null 2>&1 || true
!python -c "import sys; sys.argv=['x']"  # noop

import subprocess
import sys

cmd = [sys.executable, '-m', 'pip', 'show', 'voiceforensics']
subprocess.run(cmd, check=False)

# Use the orchestrator script if you cloned the repo; otherwise call the CLI directly:
!python -m voiceforensics train demo_dataset --model rawnet2 -o weights/rawnet2.pth --epochs 3 --batch-size 4 --device auto
!python -m voiceforensics train demo_dataset --model aasist  -o weights/aasist.pth  --epochs 3 --batch-size 4 --device auto
!python -m voiceforensics calibrate demo_dataset -o weights/calibration.env

## 5. Activate the trained weights & verify

In [ ]:
import os

os.environ['VF_RAWNET2_WEIGHTS_PATH'] = str(Path('weights/rawnet2.pth').resolve())
os.environ['VF_AASIST_WEIGHTS_PATH'] = str(Path('weights/aasist.pth').resolve())
from voiceforensics.config import get_settings; get_settings.cache_clear()
from voiceforensics.pipeline import Engine

engine = Engine()
print('active detectors:', engine.active_detector_names)
print('baseline_only:', engine.baseline_only)  # -> False once weights load

## 6. Export

Download `weights/rawnet2.pth`, `weights/aasist.pth`, and `weights/calibration.env`.
On your server set the matching `VF_*_WEIGHTS_PATH` env vars and `source calibration.env`.

In [ ]:
import shutil

shutil.make_archive('voiceforensics_weights', 'zip', 'weights')
print('created voiceforensics_weights.zip')
try:
    from google.colab import files  # type: ignore
    files.download('voiceforensics_weights.zip')
except Exception:
    print('Not on Colab — download weights/ manually.')